# CNN + RL Shortlist Runner

Clean Colab notebook for the experimental shortlist-based routing branch.
This notebook does not replace the old PPO/DQN notebooks.


In [ ]:
# 1) Sync repo and branch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BRANCH = 'elora-cnn-rl-shortlist'
REPO_URL = 'https://github.com/helloelora/rl-quantum-circuit-routing.git'
REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')

if not (REPO_DIR / '.git').exists():
    !git clone -b {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin --prune
    !git show-ref --verify --quiet refs/heads/{BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!git branch --show-current
!git log -1 --oneline


In [ ]:
# 2) Install/check dependencies
%cd {REPO_DIR}
%pip -q install "numpy==1.26.4" "qiskit==1.4.2" "gymnasium==0.29.1" "networkx>=3.2,<4" "matplotlib==3.8.3"

import numpy, qiskit, gymnasium, networkx, matplotlib, torch
print('numpy:', numpy.__version__)
print('qiskit:', qiskit.__version__)
print('gymnasium:', gymnasium.__version__)
print('networkx:', networkx.__version__)
print('matplotlib:', matplotlib.__version__)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

!python train_shortlist.py --help | head -n 60


In [ ]:
# 3) Launch recommended PPO shortlist run
from datetime import datetime

RUN_NAME = 'shortlist_ppo_' + datetime.now().strftime('%Y%m%d_%H%M%S')

!python train_shortlist.py \
  --project-root /content/drive/MyDrive/rl-quantum-circuit-routing \
  --algo ppo \
  --run-name {RUN_NAME} \
  --topologies heavy_hex_19,grid_3x3,linear_5 \
  --linear-topology-weight 0.5 \
  --grid-topology-weight 1.5 \
  --heavy-hex-topology-weight 2.0 \
  --circuit-depth 14 \
  --total-timesteps 400000 \
  --rollout-steps 4096 \
  --learning-rate 2e-4 \
  --gamma 0.99 \
  --gae-lambda 0.97 \
  --clip-range 0.15 \
  --update-epochs 8 \
  --minibatch-size 256 \
  --entropy-coef-start 0.003 \
  --entropy-coef-end 0.0001 \
  --completion-bonus 15 \
  --timeout-penalty -8 \
  --gate-reward-coeff 1.0 \
  --step-penalty -0.05 \
  --reverse-swap-penalty -0.2 \
  --repeat-swap-penalty-coeff -0.05 \
  --repeat-swap-penalty-cap -1.2 \
  --no-progress-penalty-coeff -0.02 \
  --no-progress-penalty-cap -0.8 \
  --distance-reward-coeff-start 0.03 \
  --distance-reward-coeff-end 0.015 \
  --max-steps-per-two-qubit-gate 7 \
  --max-steps-min 40 \
  --max-steps-max 320 \
  --candidate-expand-hops 1 \
  --min-two-qubit-gates 8 \
  --eval-interval-updates 20 \
  --eval-circuits-per-topology 12 \
  --eval-circuit-depth 14 \
  --eval-min-two-qubit-gates 8 \
  --trace-interval-updates 20 \
  --trace-cases-per-topology 2 \
  --trace-max-steps 220 \
  --device auto

print('RUN_NAME =', RUN_NAME)


In [ ]:
# 4) Optional DQN shortlist run
# from datetime import datetime
# RUN_NAME = 'shortlist_dqn_' + datetime.now().strftime('%Y%m%d_%H%M%S')
# !python train_shortlist.py \
#   --project-root /content/drive/MyDrive/rl-quantum-circuit-routing \
#   --algo dqn \
#   --run-name {RUN_NAME} \
#   --topologies heavy_hex_19,grid_3x3,linear_5 \
#   --linear-topology-weight 0.5 \
#   --grid-topology-weight 1.5 \
#   --heavy-hex-topology-weight 2.0 \
#   --circuit-depth 14 \
#   --total-timesteps 400000 \
#   --rollout-steps 4096 \
#   --learning-rate 2e-4 \
#   --gamma 0.99 \
#   --dqn-replay-size 200000 \
#   --dqn-min-replay-size 10000 \
#   --dqn-batch-size 256 \
#   --dqn-target-update-interval-steps 2000 \
#   --dqn-epsilon-start 0.8 \
#   --dqn-epsilon-end 0.08 \
#   --dqn-epsilon-decay-steps 280000 \
#   --completion-bonus 15 \
#   --timeout-penalty -8 \
#   --gate-reward-coeff 1.0 \
#   --step-penalty -0.05 \
#   --reverse-swap-penalty -0.2 \
#   --repeat-swap-penalty-coeff -0.05 \
#   --repeat-swap-penalty-cap -1.2 \
#   --no-progress-penalty-coeff -0.02 \
#   --no-progress-penalty-cap -0.8 \
#   --distance-reward-coeff-start 0.03 \
#   --distance-reward-coeff-end 0.015 \
#   --max-steps-per-two-qubit-gate 7 \
#   --max-steps-min 40 \
#   --max-steps-max 320 \
#   --candidate-expand-hops 1 \
#   --min-two-qubit-gates 8 \
#   --eval-interval-updates 20 \
#   --eval-circuits-per-topology 12 \
#   --eval-circuit-depth 14 \
#   --eval-min-two-qubit-gates 8 \
#   --trace-interval-updates 20 \
#   --trace-cases-per-topology 2 \
#   --trace-max-steps 220 \
#   --device auto


In [ ]:
# 5) Quick summary for one run
from pathlib import Path
import csv, json

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')
RUN_DIR = REPO_DIR / 'runs' / RUN_NAME
METRICS = RUN_DIR / 'metrics.csv'
BEST = RUN_DIR / 'best_eval_metrics.json'

print('RUN_DIR:', RUN_DIR)
print('metrics exists:', METRICS.exists())
print('best_eval exists:', BEST.exists())

if BEST.exists():
    print(json.dumps(json.loads(BEST.read_text()), indent=2))

if METRICS.exists():
    with METRICS.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    eval_rows = [r for r in rows if r.get('eval_improvement_pct', '') not in ('', 'nan', 'None')]
    trace_rows = [r for r in rows if r.get('trace_action_dom_ratio', '') not in ('', 'nan', 'None')]
    print('last eval row:', eval_rows[-1] if eval_rows else 'none')
    print('last trace row:', trace_rows[-1] if trace_rows else 'none')
